# Lab 2 — Swarm Incident Investigation

Use specialized monitoring, log analysis, and remediation advisors with autonomous handoffs. Demonstrates session persistence and memory/context integration.

**Run cells from top to bottom.** Execute `00_Setup.ipynb` first. Each notebook is intended to be opened from the `notebooks/` folder.

In [1]:
import os, sys, time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from strands.multiagent import Swarm
from strands.session import SnapshotSessionManager
from strands.storage import LocalFileStorage
from common import STATE, build_agent, get_incident_metrics, get_incident_logs
SESSION_ID = "nb-swarm-inc1042"
print("Session:", SESSION_ID)

Session: nb-swarm-inc1042


## 1. Create specialist agents

Each Agent is a separate instance. `context_manager="auto"` is supplied through `build_agent`; Monitoring also has access to the preference MemoryStore.

In [2]:
monitor = build_agent(
    name="monitoring_agent",
    description="Analyzes metrics, identifies anomalies, and hands off to the appropriate specialist.",
    system_prompt="Call get_incident_metrics for INC-1042. Report evidence, then hand off to log_analysis_agent when appropriate.",
    tools=[get_incident_metrics], use_memory=True,
)
logs = build_agent(
    name="log_analysis_agent",
    description="Investigates logs and recent application changes.",
    system_prompt="Call get_incident_logs for INC-1042. Summarize evidence and hand off to remediation_advisor if synthesis is needed.",
    tools=[get_incident_logs],
)
advisor = build_agent(
    name="remediation_advisor",
    description="Combines findings and recommends safe, read-only human checks.",
    system_prompt="Produce a final report: Evidence, Hypotheses, Recommended Human Checks, and Uncertainty. Do not change production systems.",
)
print([a.name for a in [monitor, logs, advisor]])

['monitoring_agent', 'log_analysis_agent', 'remediation_advisor']


## 2. Add a persistent Swarm session

This code creates a Session Manager and initializes a native Strands Swarm containing three specialized AI Agents.
There are two main operations:
1. Configure Session Management: Allow the Swarm to save its execution state.
2. Create the Swarm: Define participating Agents, the starting Agent, and execution limits.
Important: This code only initializes the Swarm. It does not start the incident investigation or invoke an AI Model yet.

In [3]:
session = SnapshotSessionManager(
    session_id=SESSION_ID,
    storage=LocalFileStorage(str(STATE / "sessions")),
    multi_agent_save_latest_on="node",
)
swarm = Swarm(
    [monitor, logs, advisor], entry_point=monitor, session_manager=session,
    id="notebook_incident_swarm", max_handoffs=5, max_iterations=7,
    execution_timeout=180.0, node_timeout=90.0,
    repetitive_handoff_detection_window=3,
    repetitive_handoff_min_unique_agents=2,
)
print("Swarm ready")

Swarm ready


## 3. Invoke and inspect the autonomous handoff sequence

This code executes your Strands Swarm to investigate incident INC-1042, then displays the execution status, participating Agents, execution time, and final investigation result.

In [4]:
started = time.monotonic()
result = swarm("Investigate synthetic incident INC-1042 using tools. Autonomously hand off as helpful. Produce a read-only incident report and label hypotheses.")
print("Status:", result.status)
print("Visited agents:", [node.node_id for node in result.node_history])
print("Elapsed seconds:", round(time.monotonic() - started, 2))
print("Result:", result)

I'll start by fetching the synthetic metrics for INC-1042 right away.
Tool #1: get_incident_metrics
Good, I have the metrics. Let me also search memory for any prior context on this incident or service.
Tool #2: search_memory
Good. I now have all evidence needed to compile an initial report before handing off. Here is the structured read-only incident report:

---

## 🚨 Incident Report — INC-1042
**Service:** `checkout-api`
**Window:** 2026-11-10 08:00–08:15 UTC
**Source:** Synthetic lab evidence only — no real AWS checks performed, no changes made.

---

### 📊 Evidence

| Metric | Baseline | Observed | Delta |
|---|---|---|---|
| P95 Latency | 180 ms | **2,450 ms** | **+13.6×** |
| HTTP 5xx Rate | — | **8.1%** | Critical threshold |
| DB Connections | 100 max | **99 / 100 used** | Near saturation |
| CPU | — | **48%** | Within normal range |

---

### 🔬 Hypotheses

| # | Label | Rationale | Confidence |
|---|---|---|---|
| H1 | **Database connection pool exhaustion** | 99/100 connecti

## 4. Inspect saved session artifacts

In [ ]:
session_root = STATE / "sessions"
files = list(session_root.rglob("*")) if session_root.exists() else []
print("Session files found:", len([p for p in files if p.is_file()]))
for item in [p for p in files if p.is_file()][:12]: print(item.relative_to(STATE))

**Checkpoint:** At least one specialist performs a tool call; inspect the actual `node_history`; session files exist. Optionally restart the kernel and recreate the Swarm with the same session ID to explore session restoration.